


**1. Add Calibration Metrics**




In [ ]:
from sklearn.calibration import calibration_curve



**2. Add ECE Calculation**




In [ ]:
def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins)
    ece = 0
    for i in range(n_bins-1):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if np.sum(mask) > 0:
            acc = np.mean(labels[mask] == (probs[mask] > 0.5))
            conf = np.mean(probs[mask])
            ece += np.abs(acc - conf) * np.sum(mask) / len(probs)
    return ece



**Monte Carlo Sampling (Bayesian Uncertainty)**




In [ ]:
def mc_predict(model, x, T=20):
    model.train()  # keep dropout active
    preds = [model(x) for _ in range(T)]
    return torch.mean(torch.stack(preds), dim=0), torch.std(torch.stack(preds), dim=0)




**Ablation Study Code (QABT)**




In [ ]:
# ===============================
# Ablation Study: QABT Components
# ===============================

from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

# --- Variant 1: Full Model (QABT) ---
full_model = qabt  # already trained

# --- Variant 2: Without Quantum Layer ---
class NoQuantumModel(QABT):
    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0,:]
        # directly pass classical features (no quantum)
        return self.head(h[:, :n_qubits])

no_quantum = NoQuantumModel(C).to(device)

# --- Variant 3: Without Bayesian Layer ---
class NoBayesianModel(QABT):
    def __init__(self, C):
        super().__init__(C)
        # replace Bayesian head with deterministic layer
        self.head = nn.Linear(n_qubits, C)

no_bayes = NoBayesianModel(C).to(device)

# ===============================
# Training Function
# ===============================
def train_model(model, train_loader, epochs=1):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    for ep in range(epochs):
        model.train()
        for batch in train_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(ids, mask)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

# ===============================
# Train Ablation Models
# ===============================
train_model(no_quantum, train_loader)
train_model(no_bayes, train_loader)

# ===============================
# Evaluation Function
# ===============================
def evaluate(model, test_loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in test_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels']

            logits = model(ids, mask)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            y_true.extend(labels.numpy())
            y_pred.extend(preds)

    return f1_score(y_true, y_pred, average='macro')

# ===============================
# Compute Results
# ===============================
f1_full = evaluate(full_model, test_loader)
f1_no_quantum = evaluate(no_quantum, test_loader)
f1_no_bayes = evaluate(no_bayes, test_loader)

print("Ablation Results:")
print(f"Full Model (QABT): {f1_full:.4f}")
print(f"Without Quantum : {f1_no_quantum:.4f}")
print(f"Without Bayesian: {f1_no_bayes:.4f}")

# ===============================
# Plot (Figure 4: Ablation Graph)
# ===============================
labels = ['Full QABT', '-Quantum', '-Bayesian']
scores = [f1_full, f1_no_quantum, f1_no_bayes]

plt.figure()
plt.bar(labels, scores)
plt.xlabel("Model Variant")
plt.ylabel("F1 Score")
plt.title("Ablation Study of QABT Components")
plt.show()




**Tensor Conversion Bug**




In [ ]:
torch.tensor(circuit(...))




**Testing Added**



In [ ]:
assert len(y_true) == len(y_pred)
assert cm.shape[0] == cm.shape[1]